# ReLU vs EReLU — Iris Dataset Lab

**Aim:** Implement ReLU and Elastic ReLU (EReLU) in Python, visualize their outputs, train the same neural network with both activations, and analyze convergence and classification performance.

> **EReLU definition used here:** during training, the positive branch is randomly scaled as `k*x`, where `k ~ Uniform(1-α, 1+α)`; the negative branch is `0`. During evaluation, `k=1`, so EReLU becomes standard ReLU. This follows the original Elastic ReLU formulation.


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Implement and visualize ReLU and EReLU

In [ ]:
# ReLU
def relu(x):
    return np.maximum(0, x)

# EReLU:
# During training: f(x) = k*x for x > 0, and 0 otherwise,
# where k is sampled from Uniform(1-alpha, 1+alpha).
# During evaluation: k = 1.
def erelu(x, alpha=0.2, training=True, rng=None):
    x = np.asarray(x)
    if not training:
        k = 1.0
    else:
        if rng is None:
            rng = np.random.default_rng()
        k = rng.uniform(1 - alpha, 1 + alpha)
    return np.where(x > 0, k * x, 0)

x = np.linspace(-5, 5, 500)
rng = np.random.default_rng(42)

plt.figure(figsize=(9, 5))
plt.plot(x, relu(x), label="ReLU", linewidth=2)
for i in range(4):
    plt.plot(x, erelu(x, alpha=0.2, training=True, rng=rng),
             linestyle="--", alpha=0.7, label="EReLU sample" if i == 0 else None)
plt.plot(x, erelu(x, alpha=0.2, training=False),
         linestyle=":", linewidth=3, label="EReLU during evaluation")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("Input x")
plt.ylabel("Activation output")
plt.title("ReLU vs Elastic ReLU (EReLU)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


### Conceptual comparison

| Property | ReLU | EReLU |
|---|---|---|
| Positive input | `f(x)=x` | `f(x)=k x` during training, with random `k` around 1 |
| Negative input | `0` | `0` |
| Positive gradient | `1` | `k` during training; `1` during evaluation |
| Negative gradient | `0` | `0` |
| Dead-neuron issue | Can occur because negative inputs receive zero gradient | Still possible because the negative branch is zero |
| Vanishing gradient on positive branch | No, for ordinary ReLU | No, because `k` stays near 1 |
| Extra computation | Very low | Slightly higher during training because of random scaling |
| Main idea | Fixed positive slope | Adds controlled fluctuation to the positive response during training |

**Important:** EReLU does **not** completely solve the dead-neuron problem because, like ReLU, its negative output is zero. Its main purpose is to introduce controlled response variation during training.


## 2. Prepare the Iris dataset

The Iris dataset contains 150 samples, 4 numerical features, and 3 classes. We standardize the input features using statistics computed only from the training data.


In [ ]:
iris = load_iris()
X = iris.data.astype(np.float32)
y = iris.target.astype(np.int64)

print("Dataset shape:", X.shape)
print("Classes:", iris.target_names)

# Main experiment split
TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Validation set comes from the training portion.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


In [ ]:
# Convert arrays to PyTorch tensors
Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xv = torch.tensor(X_val, dtype=torch.float32)
yv = torch.tensor(y_val, dtype=torch.long)
Xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.long)


## 3. Define ReLU and EReLU neural-network activations

The network architecture is identical in both experiments. Only the hidden-layer activation changes.

**Architecture:** `4 → 16 → 8 → 3`

Optimizer and all other hyperparameters are kept identical for a fair comparison.


In [ ]:
class ElasticReLU(nn.Module):
    def __init__(self, alpha=0.2):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        if self.training:
            # One random slope per forward pass.
            k = torch.empty(1, device=x.device).uniform_(
                1.0 - self.alpha, 1.0 + self.alpha
            )
        else:
            # Expected value of k is 1, so evaluation becomes ReLU.
            k = torch.tensor(1.0, device=x.device)
        return torch.where(x > 0, k * x, torch.zeros_like(x))


class SimpleNet(nn.Module):
    def __init__(self, activation="relu", alpha=0.2):
        super().__init__()

        if activation == "relu":
            act = nn.ReLU
        elif activation == "erelu":
            act = lambda: ElasticReLU(alpha=alpha)
        else:
            raise ValueError("activation must be 'relu' or 'erelu'")

        self.net = nn.Sequential(
            nn.Linear(4, 16),
            act(),
            nn.Linear(16, 8),
            act(),
            nn.Linear(8, 3)
        )

    def forward(self, x):
        return self.net(x)


def make_loaders(batch_size=16):
    train_loader = DataLoader(
        TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True
    )
    return train_loader


## 4. Train the same network separately with ReLU and EReLU

The training function records **training loss, validation loss, training accuracy, and validation accuracy for every epoch**.


In [ ]:
def evaluate(model, X_data, y_data, criterion):
    model.eval()
    with torch.no_grad():
        logits = model(X_data.to(device))
        loss = criterion(logits, y_data.to(device)).item()
        preds = logits.argmax(dim=1).cpu().numpy()
    acc = accuracy_score(y_data.numpy(), preds)
    return loss, acc


def train_model(activation, learning_rate=0.01, epochs=150,
                alpha=0.2, batch_size=16, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SimpleNet(activation=activation, alpha=alpha).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_loader = make_loaders(batch_size=batch_size)

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": []
    }

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        train_loss, train_acc = evaluate(model, Xtr, ytr, criterion)
        val_loss, val_acc = evaluate(model, Xv, yv, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

    return model, history


EPOCHS = 150
LEARNING_RATE = 0.01
ALPHA = 0.20

relu_model, relu_history = train_model(
    "relu", learning_rate=LEARNING_RATE, epochs=EPOCHS,
    alpha=ALPHA, seed=SEED
)

erelu_model, erelu_history = train_model(
    "erelu", learning_rate=LEARNING_RATE, epochs=EPOCHS,
    alpha=ALPHA, seed=SEED
)

print("Training completed.")


## 5. Performance analysis: loss and accuracy curves

In [ ]:
epochs = np.arange(1, EPOCHS + 1)

# Loss curves
plt.figure(figsize=(10, 5))
plt.plot(epochs, relu_history["train_loss"], label="ReLU Train Loss")
plt.plot(epochs, relu_history["val_loss"], label="ReLU Validation Loss")
plt.plot(epochs, erelu_history["train_loss"], label="EReLU Train Loss", linestyle="--")
plt.plot(epochs, erelu_history["val_loss"], label="EReLU Validation Loss", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# Accuracy curves
plt.figure(figsize=(10, 5))
plt.plot(epochs, relu_history["train_acc"], label="ReLU Train Accuracy")
plt.plot(epochs, relu_history["val_acc"], label="ReLU Validation Accuracy")
plt.plot(epochs, erelu_history["train_acc"], label="EReLU Train Accuracy", linestyle="--")
plt.plot(epochs, erelu_history["val_acc"], label="EReLU Validation Accuracy", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.ylim(0.7, 1.02)
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
# Numerical comparison
def summary_row(name, history):
    return {
        "Activation": name,
        "Final Train Loss": history["train_loss"][-1],
        "Final Val Loss": history["val_loss"][-1],
        "Final Train Accuracy": history["train_acc"][-1],
        "Final Val Accuracy": history["val_acc"][-1],
        "Best Val Accuracy": max(history["val_acc"]),
        "Epoch of Best Val Accuracy": int(np.argmax(history["val_acc"]) + 1)
    }

comparison = pd.DataFrame([
    summary_row("ReLU", relu_history),
    summary_row("EReLU", erelu_history)
])

display(comparison.round(4))


## 6. Test-set performance

In [ ]:
criterion = nn.CrossEntropyLoss()

relu_test_loss, relu_test_acc = evaluate(relu_model, Xte, yte, criterion)
erelu_test_loss, erelu_test_acc = evaluate(erelu_model, Xte, yte, criterion)

test_results = pd.DataFrame({
    "Activation": ["ReLU", "EReLU"],
    "Test Loss": [relu_test_loss, erelu_test_loss],
    "Test Accuracy": [relu_test_acc, erelu_test_acc]
})

display(test_results.round(4))

print("\nReLU classification report")
relu_model.eval()
with torch.no_grad():
    relu_pred = relu_model(Xte.to(device)).argmax(1).cpu().numpy()
print(classification_report(y_test, relu_pred, target_names=iris.target_names))

print("EReLU classification report")
erelu_model.eval()
with torch.no_grad():
    erelu_pred = erelu_model(Xte.to(device)).argmax(1).cpu().numpy()
print(classification_report(y_test, erelu_pred, target_names=iris.target_names))


## 7. Parameter analysis

We now vary:

- `test_size`
- `random_state`
- `learning_rate`
- `number of epochs`

For each setting, the same architecture is trained with both activations. The tables and plots below make it easier to compare their sensitivity.

**Note:** Because EReLU is intentionally stochastic during training, small differences between repeated runs are expected.


In [ ]:
def run_experiment(test_size=0.20, random_state=42,
                    learning_rate=0.01, epochs=100, alpha=0.2):
    # Split
    Xa_train_full, Xa_test, ya_train_full, ya_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    Xa_train, Xa_val, ya_train, ya_val = train_test_split(
        Xa_train_full, ya_train_full, test_size=0.20,
        random_state=random_state, stratify=ya_train_full
    )

    # Scale
    sc = StandardScaler()
    Xa_train = sc.fit_transform(Xa_train).astype(np.float32)
    Xa_val = sc.transform(Xa_val).astype(np.float32)
    Xa_test = sc.transform(Xa_test).astype(np.float32)

    # Temporarily use local tensors in a dedicated trainer
    Xtr_old, ytr_old, Xv_old, yv_old = Xtr, ytr, Xv, yv

    globals()["Xtr"], globals()["ytr"] = (
        torch.tensor(Xa_train, dtype=torch.float32),
        torch.tensor(ya_train, dtype=torch.long)
    )
    globals()["Xv"], globals()["yv"] = (
        torch.tensor(Xa_val, dtype=torch.float32),
        torch.tensor(ya_val, dtype=torch.long)
    )

    out = []
    for activation in ["relu", "erelu"]:
        model, hist = train_model(
            activation, learning_rate=learning_rate,
            epochs=epochs, alpha=alpha, seed=random_state
        )

        criterion = nn.CrossEntropyLoss()
        test_loss, test_acc = evaluate(
            model,
            torch.tensor(Xa_test, dtype=torch.float32),
            torch.tensor(ya_test, dtype=torch.long),
            criterion
        )

        out.append({
            "Activation": activation,
            "Test Loss": test_loss,
            "Test Accuracy": test_acc,
            "Best Validation Accuracy": max(hist["val_acc"]),
            "Best Validation Epoch": int(np.argmax(hist["val_acc"]) + 1)
        })

    globals()["Xtr"], globals()["ytr"], globals()["Xv"], globals()["yv"] = (
        Xtr_old, ytr_old, Xv_old, yv_old
    )

    return pd.DataFrame(out)


### 7A. Vary `test_size`

In [ ]:
test_sizes = [0.15, 0.20, 0.25, 0.30]
rows = []

for ts in test_sizes:
    r = run_experiment(test_size=ts, random_state=42,
                       learning_rate=0.01, epochs=100, alpha=0.20)
    r["test_size"] = ts
    rows.append(r)

test_size_results = pd.concat(rows, ignore_index=True)
display(test_size_results.round(4))

plt.figure(figsize=(9, 5))
for act in ["relu", "erelu"]:
    d = test_size_results[test_size_results["Activation"] == act]
    plt.plot(d["test_size"], d["Test Accuracy"], marker="o", label=act.upper())
plt.xlabel("test_size")
plt.ylabel("Test Accuracy")
plt.title("Effect of test_size")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


### 7B. Vary `random_state`

In [ ]:
random_states = [0, 7, 21, 42, 99]
rows = []

for rs in random_states:
    r = run_experiment(test_size=0.20, random_state=rs,
                       learning_rate=0.01, epochs=100, alpha=0.20)
    r["random_state"] = rs
    rows.append(r)

random_state_results = pd.concat(rows, ignore_index=True)
display(random_state_results.round(4))

plt.figure(figsize=(9, 5))
for act in ["relu", "erelu"]:
    d = random_state_results[random_state_results["Activation"] == act]
    plt.plot(d["random_state"], d["Test Accuracy"], marker="o", label=act.upper())
plt.xlabel("random_state")
plt.ylabel("Test Accuracy")
plt.title("Effect of random_state")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


### 7C. Vary learning rate

In [ ]:
learning_rates = [0.001, 0.005, 0.01, 0.05]
rows = []

for lr in learning_rates:
    r = run_experiment(test_size=0.20, random_state=42,
                       learning_rate=lr, epochs=100, alpha=0.20)
    r["learning_rate"] = lr
    rows.append(r)

lr_results = pd.concat(rows, ignore_index=True)
display(lr_results.round(4))

plt.figure(figsize=(9, 5))
for act in ["relu", "erelu"]:
    d = lr_results[lr_results["Activation"] == act]
    plt.plot(d["learning_rate"], d["Test Accuracy"], marker="o", label=act.upper())
plt.xlabel("Learning Rate")
plt.ylabel("Test Accuracy")
plt.title("Effect of Learning Rate")
plt.xscale("log")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


### 7D. Vary number of epochs

In [ ]:
epoch_values = [25, 50, 100, 200]
rows = []

for ep in epoch_values:
    r = run_experiment(test_size=0.20, random_state=42,
                       learning_rate=0.01, epochs=ep, alpha=0.20)
    r["epochs"] = ep
    rows.append(r)

epoch_results = pd.concat(rows, ignore_index=True)
display(epoch_results.round(4))

plt.figure(figsize=(9, 5))
for act in ["relu", "erelu"]:
    d = epoch_results[epoch_results["Activation"] == act]
    plt.plot(d["epochs"], d["Test Accuracy"], marker="o", label=act.upper())
plt.xlabel("Number of Epochs")
plt.ylabel("Test Accuracy")
plt.title("Effect of Number of Epochs")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 8. Automatic conclusion based on the main experiment

The conclusion below is generated from the actual results produced in this notebook, rather than assuming beforehand that one activation must win.


In [ ]:
relu_best_val = max(relu_history["val_acc"])
erelu_best_val = max(erelu_history["val_acc"])

print("===== CONCLUSION =====")
if relu_test_acc > erelu_test_acc:
    winner = "ReLU"
elif erelu_test_acc > relu_test_acc:
    winner = "EReLU"
else:
    winner = "Tie"

print(f"Main experiment test-accuracy winner: {winner}")
print(f"ReLU  - final validation accuracy: {relu_history['val_acc'][-1]:.4f}, "
      f"best validation accuracy: {relu_best_val:.4f}, test accuracy: {relu_test_acc:.4f}")
print(f"EReLU - final validation accuracy: {erelu_history['val_acc'][-1]:.4f}, "
      f"best validation accuracy: {erelu_best_val:.4f}, test accuracy: {erelu_test_acc:.4f}")

print("\nInterpretation:")
print("- ReLU is simpler and has a fixed positive slope of 1.")
print("- EReLU randomly perturbs the positive slope during training and becomes ReLU during evaluation.")
print("- On a small, relatively easy dataset such as Iris, either activation may achieve very high accuracy.")
print("- If EReLU is better, the controlled fluctuation may have improved robustness/generalization.")
print("- If ReLU is better, the extra stochasticity was not useful for this small dataset and the simpler activation was sufficient.")
print("- Therefore, the final lab conclusion should be based on the numerical results printed above.")


## 9. Short viva-ready answers

**Why is ReLU popular?**  
It is simple, fast, and has a non-zero constant gradient for positive inputs, which generally makes optimization easier than saturating activations.

**What is the main difference between ReLU and EReLU?**  
ReLU uses a fixed positive slope of 1. EReLU randomly changes the positive slope around 1 during training, then uses slope 1 during evaluation.

**Does EReLU eliminate dead neurons?**  
No. Both functions output zero for negative inputs, so EReLU can still have inactive neurons.

**Why standardize Iris features?**  
Neural networks are sensitive to feature scale. Standardization puts the input features on comparable scales and can improve optimization.

**Why keep the architecture and hyperparameters identical?**  
To make the activation function the main experimental difference and produce a fair comparison.


### Reference notes

The notebook follows the documented scikit-learn workflow for loading Iris, splitting data, and standardizing features. The EReLU implementation follows the published Elastic ReLU idea in which the positive branch is randomly scaled during training and becomes standard ReLU at test time.
